# Week 32, Day 4: Active dendrites and dendritic computation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A branch's response to its synapses

Write `B(V, Mg=1.0)`. Then, for a branch compartment with leak 10 nS and synapses each adding 0.2 nS of AMPA and 3 nS of NMDA conductance (all reversing at 0 mV, held open for the simulation), compute the steady depolarisation for 0 to 20 active synapses, with 1 mM and 0 mM magnesium.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def B(V, Mg=1.0):
    return ...

def steady(n, Mg=1.0, gL=10.0, gA=0.2, gN=3.0, C=30.0, T=300.0, dt=0.02):
    V = -70.0
    for _ in range(int(T / dt)):
        V += dt * (-gL * (V + 70) - n * gA * V - n * gN * B(V, Mg) * V) / C
    return V + 70                                   # depolarisation from rest (mV)

ns = list(range(0, 21, 2))
resp = {Mg: [steady(n, Mg) for n in ns] for Mg in (1.0, 0.0)}
for n, a, b in zip(ns, resp[1.0], resp[0.0]):
    print(f"{n:2d} synapses: {a:5.1f} mV with magnesium, {b:5.1f} mV without")
plt.plot(ns, resp[1.0], "o-", label="1 mM Mg"); plt.plot(ns, resp[0.0], "s-", label="no Mg")
plt.xlabel("active synapses on the branch"); plt.ylabel("depolarisation (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_block():
    "B gives the unblocked fraction: about 0.06 at -70 mV and above 0.9 at 0 mV with 1 mM magnesium"
    assert 0.03 < B(-70.0) < 0.1 and B(0.0) > 0.7 and abs(B(-70.0, 0.0) - 1) < 1e-12

def test_shape():
    "with magnesium the response is supralinear at first (the step from 6 to 8 synapses exceeds the step from 0 to 2) and without it is sublinear"
    with_mg, no_mg = resp[1.0], resp[0.0]
    assert with_mg[4] - with_mg[3] > with_mg[1] - with_mg[0] and no_mg[4] - no_mg[3] < no_mg[1] - no_mg[0]

test_block()
test_shape()
print("All checks passed")

### Exercise: Where the synapses are

Define `f(n)` by linear interpolation of the with-magnesium responses in `resp[1.0]` over `ns`. With an attenuation of 0.3 from each branch to the soma, compute the somatic depolarisation for twelve synapses all on one branch and for three on each of four branches, and store them in `soma`.

In [ ]:
def f(n):
    return ...

atten = 0.3
soma = {"clustered": atten * (f(12) + 3 * f(0)), "dispersed": ...}
print(f"clustered: {soma['clustered']:.1f} mV at the soma; dispersed: {soma['dispersed']:.1f} mV")
for k in (4, 8, 12, 16):
    print(f"{k:2d} synapses: clustered {atten * f(k):5.1f} mV, spread over 4 branches {atten * 4 * f(k / 4):5.1f} mV")

In [ ]:
import numpy as np
def test_f():
    "f interpolates the branch response computed above"
    assert abs(f(4) - resp[1.0][2]) < 1e-9 and abs(f(0)) < 1e-9

def test_cluster():
    "twelve synapses clustered on one branch drive the soma more than three on each of four branches"
    assert soma["clustered"] > 1.2 * soma["dispersed"]

test_f()
test_cluster()
print("All checks passed")

### Exercise: A function no point neuron can compute

X holds the four patterns (a, b, c, d) and Y the desired outputs. Try 20,000 random weight vectors and thresholds for a point neuron (fire if w · x exceeds θ) and record the most cases any of them gets right in `best_point`. Then write `two_layer(x)`: each branch fires if its two inputs sum to 2, and the soma fires if any branch fires.

In [ ]:
X = np.array([[1, 1, 0, 0], [0, 0, 1, 1], [1, 0, 1, 0], [0, 1, 0, 1]])
Y = np.array([1, 1, 0, 0])
r = np.random.default_rng(32)
best_point = 0
for _ in range(20000):
    w, theta = r.normal(size=4), r.normal()
    correct = ...
    best_point = max(best_point, correct)

def two_layer(x):
    branch1 = ...
    branch2 = ...
    return int(branch1 or branch2)

print(f"best point neuron: {best_point} of 4 correct;  two-layer: {[two_layer(x) for x in X]} against {Y.tolist()}")

In [ ]:
import numpy as np
def test_point():
    "no random linear threshold unit solves the task"
    assert best_point < 4

def test_two_layer():
    "the two-layer model with branch thresholds solves all four cases"
    assert all(two_layer(x) == y for x, y in zip(X, Y))

test_point()
test_two_layer()
print("All checks passed")